In [1]:
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import os
from datasets import load_dataset
import torch
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers
from utils import extract_layer_mlp
from models import ParallelMLPs

notebook_dir = '/u/eboix/moe_distillation'
activ_dir = '/work/nvme/bbjr/eboix/saved_activations'
if os.path.exists(notebook_dir):
    os.chdir(notebook_dir)

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [2]:
# Choose a Pythia model size (options include: 70m, 160m, 410m, 1b, 1.4b, 2.8b, 6.9b, 12b)
model_name = "EleutherAI/pythia-410m"
layer_idx = 12
train_activ_file = activ_dir + '/wikitext_wikitext-2-raw-v1_EleutherAI_pythia-410m_layer12_actinput_train_mintok20.pt'
val_activ_file = activ_dir + '/wikitext_wikitext-2-raw-v1_EleutherAI_pythia-410m_layer12_actinput_validation_mintok20.pt'

# Load the model and tokenizer
teacher_mlp = AutoModelForCausalLM.from_pretrained(model_name).to(device)
teacher_mlp = extract_layer_mlp(teacher_mlp, layer_idx)
teacher_mlp = teacher_mlp.to(device)
teacher_mlp.eval()  # Set the model to evaluation mode

# 1. Load the activation data
train_activ = torch.load(train_activ_file)
val_activ = torch.load(val_activ_file)

# 2. Wrap into TensorDataset
train_dataset = TensorDataset(train_activ)
val_dataset = TensorDataset(val_activ)

# 3. Create DataLoader with appropriate settings
batch_size = 512 # Batch size can be adjusted; float16 uses less memory
shuffle_data = True
num_workers = 0 # Adjust based on your machine's CPU cores
pin_memory = True # Enable for faster CPU to GPU transfers

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=shuffle_data,
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=True # Keeps the last batch even if it is smaller
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=shuffle_data,
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=True # Keeps the last batch even if it is smaller
)

# Example of how to iterate and move data to GPU
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    print("CUDA not available, using CPU. GPU transfer will not occur.")

print(f"Using device: {device}")
print(f"DataLoader configured with batch_size={batch_size}, shuffle={shuffle_data}, num_workers={num_workers}, pin_memory={pin_memory}")


Using device: cuda
DataLoader configured with batch_size=512, shuffle=True, num_workers=0, pin_memory=True


In [3]:
def compute_val_activations_variance(teacher_mlp, val_dataloader, device):
    teacher_mlp.eval()
    all_activations = []

    if hasattr(teacher_mlp, 'dense_4h_to_h'):
        # If the model has a dense layer, we can get the output dimension
        out_dim = teacher_mlp.dense_4h_to_h.out_features
    else:
        raise NotImplementedError("Could not figure out output dimension of the MLP. Please check the model architecture.")

    # Compute mean of activations
    mean_activations = torch.zeros(out_dim, device=device, dtype=torch.float32)
    with torch.no_grad():
        for batch in tqdm(val_dataloader, desc="Computing activations"):
            assert(len(batch) == 1), "Batch should contain only one tensor of activations"
            batch = batch[0]
            inputs = batch.float().to(device)
            activations = teacher_mlp(inputs)
            mean_activations += torch.mean(activations, dim=0)
        mean_activations /= len(val_dataloader)
    
    mean_activations = mean_activations.unsqueeze(0)

    # Compute variance of activations
    variance = 0
    for batch in tqdm(val_dataloader, desc="Computing variance of activations"):
        assert(len(batch) == 1), "Batch should contain only one tensor of activations"
        batch = batch[0]
        inputs = batch.float().to(device)
        activations = teacher_mlp(inputs)
        variance += torch.mean((activations - mean_activations) ** 2)
    variance /= len(val_dataloader)
        
    return variance

In [ ]:
input_dim = teacher_mlp.dense_h_to_4h.in_features
output_dim = input_dim

# # Initialize the student model
# student_model = ParallelMLPs(
#     input_dim=input_dim,
#     output_dim=output_dim,
#     intermediate_dim=20,
#     multi_index_dim=4,  # Number of indices in the multi-index
#     m=2000,  # Number of parallel MLPs
#     top_k=20,  # Set to None for now, as we are not using top-k
#     bias=True,  # Bias is not yet supported in this implementation
# ).to(device)
# Initialize the student model
student_model = ParallelMLPs(
    input_dim=input_dim,
    output_dim=output_dim,
    intermediate_dim=128,
    multi_index_dim=4,  # Number of indices in the multi-index
    m=120,  # Number of parallel MLPs
    top_k=120,  # Set to None for now, as we are not using top-k
    bias=True,  # Bias is not yet supported in this implementation
).to(device)
# Print model summary
print(f"Initialized student model with {sum(p.numel() for p in student_model.parameters())/1e6:.2f}M parameters")

num_epochs = 100


# Training parameters
optimizer = optim.AdamW(student_model.parameters(), lr=1e-3)
# Add cosine learning rate scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-5)
criterion = nn.MSELoss()

val_activations_variance = compute_val_activations_variance(teacher_mlp, val_dataloader, device)

def validate_model():
    # Validate the model
    with torch.no_grad():
        val_loss = 0.0
        for batch in tqdm(val_dataloader, desc="Validating"):
            assert(len(batch) == 1), "Batch should contain only one tensor of activations"
            batch = batch[0]
            inputs = batch.float().to(device)
            student_outputs = student_model(inputs)
            teacher_outputs = teacher_mlp(inputs)
            loss = criterion(student_outputs, teacher_outputs)
            val_loss += loss.item()
        val_loss /= len(val_dataloader)
        print(f"Epoch {epoch+1}/{num_epochs}, Validation Loss: {val_loss:.4f}")
        print('Validation Loss fraction of variance:', val_loss / val_activations_variance.item())


for epoch in range(num_epochs):
    avg_loss = 0.0
    student_model.train()
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for i, batch in enumerate(progress_bar):
        if i % 1000 == 999:
            validate_model()
        assert(len(batch) == 1), "Batch should contain only one tensor of activations"
        batch = batch[0]
        inputs = batch.float().to(device)
        # Forward pass through the student model
        student_outputs = student_model(inputs)
        # Forward pass through the teacher model
        with torch.no_grad():
            teacher_outputs = teacher_mlp(inputs)
        loss = criterion(student_outputs, teacher_outputs)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        avg_loss += loss.item()
        avg_loss_so_far = avg_loss/(i+1)
        progress_bar.set_postfix({"avgloss": f"{avg_loss_so_far:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.6f}", 'loss' : f"{loss.item():.4f}"})

    avg_loss /= len(train_dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Average Train Loss: {avg_loss:.4f}")
    scheduler.step()
    


Initialized student model with 1.12M parameters


Epoch 1/100:  12%|███▎                         | 1003/8666 [00:34<49:10,  2.60it/s, avgloss=0.0121, lr=0.010000, loss=0.0107]

Epoch 1/100, Validation Loss: 0.0106
Validation Loss fraction of variance: 0.3226379292115188


Epoch 1/100:  17%|████▉                        | 1479/8666 [00:48<03:37, 33.11it/s, avgloss=0.0115, lr=0.010000, loss=0.0097]

In [ ]:
import os
import time

# Create a directory for saving models if it doesn't exist
save_dir = os.path.join(root_folder, 'saved_models')
os.makedirs(save_dir, exist_ok=True)

# Create a timestamp and model identifier for the filename
timestamp = time.strftime("%Y%m%d-%H%M%S")
model_type = "parallel_mlps"
layer_info = f"layer{layer_idx}"
model_params = f"m{student_model.m}_inter{student_model.intermediate_dim}_multi{student_model.multi_index_dim}"
loss_info = f"loss{avg_val_loss:.8f}"

# Construct filename
filename = f"{model_type}_{layer_info}_{model_params}_{loss_info}_{timestamp}.pt"
save_path = os.path.join(save_dir, filename)

# Save the model state dict and configuration
model_info = {
    'state_dict': student_model.state_dict(),
    'config': {
        'input_dim': student_model.input_dim,
        'output_dim': student_model.output_dim,
        'intermediate_dim': student_model.intermediate_dim,
        'multi_index_dim': student_model.multi_index_dim,
        'm': student_model.m,
        'top_k': student_model.top_k,
        'bias': student_model.bias
    },
    'training_info': {
        'val_loss': avg_val_loss,
        'train_loss': avg_loss,
        'teacher_model': model_name,
        'layer_idx': layer_idx,
        'variance': val_activations_variance,
        'fraction_variance': avg_val_loss / val_activations_variance
    }
}

# Save the model
torch.save(model_info, save_path)
print(f"Model saved to: {save_path}")

# Function to load the model (for reference)
def load_parallel_mlp(path):
    checkpoint = torch.load(path)
    config = checkpoint['config']
    model = ParallelMLPs(
        input_dim=config['input_dim'],
        output_dim=config['output_dim'],
        intermediate_dim=config['intermediate_dim'],
        multi_index_dim=config['multi_index_dim'],
        m=config['m'],
        top_k=config['top_k'],
        bias=config['bias']
    )
    model.load_state_dict(checkpoint['state_dict'])
    return model, checkpoint['training_info']

print("Example code to load this model:")
print("model, training_info = load_parallel_mlp(save_path)")

Model saved to: /u/eboix/moe_distillation/saved_models/parallel_mlps_layer3_m3200_inter4_multi2_loss0.00000448_20250528-155425.pt
Example code to load this model:
model, training_info = load_parallel_mlp(save_path)


In [ ]:
type(model.gpt_neox.layers[0].mlp)

transformers.models.gpt_neox.modeling_gpt_neox.GPTNeoXMLP

In [ ]:
#  Teacher model is the MLP of the specified layer
teacher_mlp = model.gpt_neox.layers[layer_idx].mlp
input_dim = teacher_mlp.dense_h_to_4h.in_features
output_dim = input_dim

# Initialize the student model
# student_model = nn.Sequential(
#     nn.Linear(input_dim, 2048),
#     nn.GELU(),
#     nn.Linear(2048, output_dim)
# ).to(device)
student_model = nn.Sequential(
    nn.Linear(input_dim, 256),
    nn.GELU(),
    nn.Linear(256, output_dim)
).to(device)

# student_model = transformers.models.gpt_neox.modeling_gpt_neox.GPTNeoXMLP(model.config).to(device)

num_train_batches_per_epoch = 100  # Number of batches to train on
num_epochs = 100

# Training parameters
optimizer = optim.AdamW(student_model.parameters(), lr=3e-4)
# Add cosine learning rate scheduler
total_steps = num_train_batches_per_epoch * num_epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
criterion = nn.MSELoss()

# Compute variance of teacher model output on activation validation set
mean_val_output = torch.zeros(output_dim, device=device)
val_activations_variance = 0.0
for val_activation in val_activations:
    with torch.no_grad():
        teacher_output = teacher_mlp(val_activation)
        mean_val_output += teacher_output.mean(dim=0)
mean_val_output /= len(val_activations)
for val_activation in val_activations:
    with torch.no_grad():
        teacher_output = teacher_mlp(val_activation)
        val_activations_variance += torch.sum((teacher_output - mean_val_output.view(1,-1)) ** 2).item() / teacher_output.shape[0]
val_activations_variance /= len(val_activations)
val_activations_variance /= mean_val_output.shape[0]  # Normalize by output dimension
print(f"Teacher model output variance on validation set: {val_activations_variance:.6f}")

# Training loop
for epoch in range(num_epochs):
    # Training phase
    total_loss = 0.0
    for _ in range(num_train_batches_per_epoch):
        try:
            layer_input = next(train_iterator)
        except StopIteration:
            train_iterator = iter(train_activation_loader)
            layer_input = next(train_iterator)
        with torch.no_grad():
            layer_output = teacher_mlp(layer_input)
        
        student_output = student_model(layer_input)
        loss = criterion(student_output, layer_output)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()  # Update learning rate
            
        total_loss += loss.item()
    avg_loss = total_loss / num_train_batches_per_epoch
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_loss:.6f}, LR: {scheduler.get_last_lr()[0]:.6f}")
    
    # Validation phase
    tot_val_loss = 0.0
    for val_activation in val_activations:
        with torch.no_grad():
            teacher_output = teacher_mlp(val_activation)
            student_output = student_model(val_activation)
            val_loss = criterion(student_output, teacher_output).item()
        tot_val_loss += val_loss
    avg_val_loss = tot_val_loss / len(val_activations)
    print(f"Validation Loss: {avg_val_loss:.6f}")
    print(f'Fraction validation loss over variance: {avg_val_loss / val_activations_variance:.6f}')

Teacher model output variance on validation set: 0.126783


/tmp/ipykernel_1445851/691209243.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


RuntimeError: mat1 and mat2 must have the same dtype, but got Half and Float

In [6]:
# Define MLP class for MoE experts
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=32):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.fc(x)

# Define MoE with Top-K (where each expert is an MLP)
class MoE_TopK(nn.Module):
    def __init__(self, input_dim, output_dim, num_experts, k, hidden_dim=32,bias=False):
        super().__init__()
        self.num_experts = num_experts
        self.k = k  # Number of selected experts

        # Gating network
        self.gate = nn.Linear(input_dim, num_experts,bias=bias)

        self.expert_tally = nn.Parameter(torch.zeros(self.num_experts))
        self.expert_tally.requires_grad = False

        # Expert networks (each expert is an MLP)
        self.experts = nn.ModuleList([MLP(input_dim, output_dim, hidden_dim) for _ in range(num_experts)])

    def forward(self, x):
        gate_scores = self.gate(x)  # (batch_size, num_experts) 
        topk_vals, topk_idxs = torch.topk(gate_scores, self.k, dim=-1)  # Get top-k expert indices
        # print(topk_idxs.shape)
        # Count the number of occurrences of each index in topk_idxs
        # unique_idxs, counts = torch.unique(topk_idxs, return_counts=True)
        # print(unique_idxs)
        # print(counts)
        # print(self.expert_tally)
        # self.expert_tally[unique_idxs] += counts
        # print('before',self.expert_tally)
        # print(topk_idxs.shape)
        # for i in range(topk_idxs.shape[0]):
        #     for j in range(topk_idxs.shape[1]):
        #         print(i,j)
        #         self.expert_tally[topk_idxs[j]] += 1
        # print('after',self.expert_tally)
        topk_weights = torch.softmax(topk_vals, dim=-1)  # Normalize weights over top-k

        batch_size, _ = x.shape
        expert_outputs = torch.stack([self.experts[i](x) for i in range(self.num_experts)], dim=1)  # (batch_size, num_experts, output_dim)
        selected_expert_outputs = torch.gather(expert_outputs, 1, topk_idxs.unsqueeze(-1).expand(-1, -1, expert_outputs.shape[-1]))

        output = torch.sum(selected_expert_outputs * topk_weights.unsqueeze(-1), dim=1)
        return output

In [7]:
# Teacher model is the MLP of the specified layer
teacher_mlp = model.gpt_neox.layers[layer_idx].mlp
input_dim = teacher_mlp.dense_h_to_4h.in_features
output_dim = input_dim

# Initialize the student model as a MoE model
num_experts = 256
k = 8  # Number of active experts
hidden_dim = 32  # Smaller hidden dimension for each expert

# Create MoE student model
student_model = MoE_TopK(
    input_dim=input_dim, 
    output_dim=output_dim, 
    num_experts=num_experts, 
    k=k,
    hidden_dim=hidden_dim
).to(device)

num_train_batches_per_epoch = 100  # Number of batches to train on
num_epochs = 100

# Training parameters
optimizer = optim.AdamW(student_model.parameters(), lr=3e-3)
# Add cosine learning rate scheduler
total_steps = num_train_batches_per_epoch * num_epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
criterion = nn.MSELoss()

# Compute variance of teacher model output on activation validation set
mean_val_output = torch.zeros(output_dim, device=device)
val_activations_variance = 0.0
for val_activation in val_activations:
    with torch.no_grad():
        teacher_output = teacher_mlp(val_activation)
        mean_val_output += teacher_output.mean(dim=0)
mean_val_output /= len(val_activations)
for val_activation in val_activations:
    with torch.no_grad():
        teacher_output = teacher_mlp(val_activation)
        val_activations_variance += torch.sum((teacher_output - mean_val_output.view(1,-1)) ** 2).item() / teacher_output.shape[0]
val_activations_variance /= len(val_activations)
val_activations_variance /= mean_val_output.shape[0]  # Normalize by output dimension
print(f"Teacher model output variance on validation set: {val_activations_variance:.6f}")

# Training loop
for epoch in range(num_epochs):
    # Training phase
    total_loss = 0.0
    for _ in range(num_train_batches_per_epoch):
        try:
            layer_input = next(train_iterator)
        except StopIteration:
            train_iterator = iter(train_activation_loader)
            layer_input = next(train_iterator)
        with torch.no_grad():
            layer_output = teacher_mlp(layer_input)
        
        student_output = student_model(layer_input)
        loss = criterion(student_output, layer_output)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()  # Update learning rate
            
        total_loss += loss.item()
    avg_loss = total_loss / num_train_batches_per_epoch
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_loss:.6f}, LR: {scheduler.get_last_lr()[0]:.6f}")
    
    # Validation phase
    tot_val_loss = 0.0
    for val_activation in val_activations:
        with torch.no_grad():
            teacher_output = teacher_mlp(val_activation)
            student_output = student_model(val_activation)
            val_loss = criterion(student_output, teacher_output).item()
        tot_val_loss += val_loss
    avg_val_loss = tot_val_loss / len(val_activations)
    print(f"Validation Loss: {avg_val_loss:.6f}")
    print(f'Fraction validation loss over variance: {avg_val_loss / val_activations_variance:.6f}')

Teacher model output variance on validation set: 0.126610
Epoch [1/100], Train Loss: 0.056560, LR: 0.002999
Validation Loss: 0.029504
Fraction validation loss over variance: 0.233028
Epoch [2/100], Train Loss: 0.026603, LR: 0.002997
Validation Loss: 0.024265
Fraction validation loss over variance: 0.191648
Epoch [3/100], Train Loss: 0.023398, LR: 0.002993
Validation Loss: 0.022290
Fraction validation loss over variance: 0.176054
Epoch [4/100], Train Loss: 0.022002, LR: 0.002988
Validation Loss: 0.021233
Fraction validation loss over variance: 0.167705
Epoch [5/100], Train Loss: 0.021188, LR: 0.002982
Validation Loss: 0.020674
Fraction validation loss over variance: 0.163286
Epoch [6/100], Train Loss: 0.020487, LR: 0.002973
Validation Loss: 0.019985
Fraction validation loss over variance: 0.157845
Epoch [7/100], Train Loss: 0.020130, LR: 0.002964
Validation Loss: 0.019659
Fraction validation loss over variance: 0.155269
Epoch [8/100], Train Loss: 0.019840, LR: 0.002953
Validation Loss: 

In [ ]:

# from old.activation_buffer import ActivationDataLoader

# # Example of how to use this class:
# # Process the dataset in batches
# batch_size = 1024
# layer_idx=3
# num_batches_in_val = 100  # Number of batches to collect from validation set

# # model = model.half()
# dtype = torch.float32

# train_text_dataloader = DataLoader(
#     dataset['train'], 
#     batch_size=1, 
#     shuffle=True, 
#     collate_fn=lambda x: [item['text'] for item in x]
# )

# val_text_dataloader = DataLoader(
#     dataset['validation'], 
#     batch_size=1, 
#     shuffle=True, 
#     collate_fn=lambda x: [item['text'] for item in x]
# )

# # Create the activation data loader
# train_activation_loader = ActivationDataLoader(
#     model=model,
#     tokenizer=tokenizer,
#     layer_idx=layer_idx,
#     max_length=None,
#     batch_size=batch_size,
#     activation_type='input',
#     text_dataloader=train_text_dataloader,
#     max_buffer_size=1000000,  # Adjust buffer size as needed
#     dtype = dtype,
# )

# # Create the activation data loader
# val_activation_loader = ActivationDataLoader(
#     model=model,
#     tokenizer=tokenizer,
#     layer_idx=layer_idx,
#     max_length=None,
#     batch_size=batch_size,
#     activation_type='input',
#     text_dataloader=val_text_dataloader,
#     max_buffer_size=100000,  # Adjust buffer size as needed
#     dtype = dtype,
# )
# train_iterator = iter(train_activation_loader)
# val_iterator = iter(val_activation_loader)
# # Create a dataset from 100 batches of the validation iterator
# val_activations = []
# for _ in tqdm(range(num_batches_in_val), desc="Collecting validation activations"):
#     try:
#         val_activations.append(next(val_iterator))
#     except StopIteration:
#         break
# del val_iterator
# del val_activation_loader


# # # Concatenate all activations
# # all_activations = np.vstack([act.reshape(-1, act.shape[-1]) for act in all_activations])
# # print(f"Collected activations shape: {all_activations.shape}")
